# 01 — 异步 LLM 客户端

这门课要做的东西是一个 **AI Coding Agent**，就是 Claude Code / Cursor 那种——你说一句话，它能自己读文件、写代码、搜索资料、执行工具，直到任务完成。

任何 Agent 的底座都是一个 **LLM 客户端**——负责把消息发给模型，拿回回答。

这个 Notebook 做这一件事：**写一个可用于生产的异步 LLM 客户端，对接本地 Ollama。**

---

## 学完这章你能做到

- 理解 messages 列表的结构，知道为什么要这样传历史记录
- 用 OpenAI SDK 对接任何兼容接口（Ollama / OpenRouter / Anthropic 都一样换个 URL）
- 实现流式输出，字一个一个打印出来而不是等全部生成
- 加指数退避重试，遇到限速或网络抖动自动重试

## 前置：Ollama 是什么

Ollama 是一个在本地运行大模型的工具。它在你电脑上起了一个 HTTP 服务器，默认监听 `http://localhost:11434`。

```
你的代码
   │
   │  HTTP 请求
   ▼
Ollama 服务器 (localhost:11434)
   │
   │  调用本地模型
   ▼
gpt-oss:120b（跑在你的 GPU 上）
```

关键点：**Ollama 的 API 格式和 OpenAI 完全一样**。这意味着用 OpenAI 的 Python SDK，改一个参数（`base_url`），就能直接对接 Ollama。

这也是为什么市面上大多数 LLM 工具都支持「OpenAI 兼容接口」——行业事实标准。

## 0. 安装依赖

In [ ]:
# 如果没装过，运行这行
# !pip install openai

In [ ]:
# 验证 Ollama 在跑
import urllib.request
try:
    with urllib.request.urlopen("http://localhost:11434") as r:
        print("✓ Ollama 正在运行")
except Exception as e:
    print(f"✗ Ollama 没有运行，请先执行 `ollama serve` 或打开 Ollama 应用")
    print(f"  错误: {e}")

## 1. messages 是什么

在讲代码之前，先把这个概念搞清楚——**后面所有代码都围绕这个格式转**。

LLM 本身是**无状态的**。它不记得你上次说了什么。每次你发请求，都必须把完整的对话历史一起带过去。

对话历史是一个列表，每条消息是一个字典：

```python
messages = [
    {"role": "system",    "content": "你是一个代码助手"},   # 系统提示，定义 AI 的行为
    {"role": "user",      "content": "帮我写一个 hello world"},  # 用户说的话
    {"role": "assistant", "content": "好的，这是代码：..."},  # AI 上次的回复
    {"role": "user",      "content": "改成 Python 版本"},    # 用户继续说
]
```

`role` 只有三种：
- `system`：定义 AI 的角色和规则，放在列表最前面，每次都带着
- `user`：用户的输入
- `assistant`：AI 的输出（历史记录）

后面还有 `tool`——工具调用的结果，我们在 Module 3 再讲。

## 2. 最简单的调用

In [ ]:
from openai import OpenAI

# 唯一和「正常」OpenAI 用法不同的地方：指定 base_url 和 api_key
# Ollama 不需要真实的 API key，随便填一个字符串就行
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # 不验证，填什么都行
)

response = client.chat.completions.create(
    model="gpt-oss:120b",
    messages=[
        {"role": "user", "content": "用一句话解释什么是递归"}
    ]
)

# 拿到回答
print(response.choices[0].message.content)

In [ ]:
# 看看完整的 response 结构
print("模型:", response.model)
print("finish_reason:", response.choices[0].finish_reason)  # stop = 正常结束
print()
print("token 使用:")
print(f"  prompt_tokens:     {response.usage.prompt_tokens}")     # 输入消耗
print(f"  completion_tokens: {response.usage.completion_tokens}") # 输出消耗
print(f"  total_tokens:      {response.usage.total_tokens}")      # 合计

**`finish_reason` 有几个值要记住：**
- `stop` — 正常生成完毕
- `length` — 达到 `max_tokens` 限制被截断了
- `tool_calls` — 模型想调用工具（后面章节会遇到）
- `null` / `None` — 还在流式输出中（流式时每个 chunk 都是 null，最后一个才是 stop）

## 3. 为什么要用异步

Agent 在运行时，一次循环里可能：
- 调用 LLM 等回答（耗时 5-30 秒）
- 同时要处理多个子代理的请求
- 要能随时响应用户的中断信号

同步代码里，等待 LLM 响应的时候整个程序就**卡死**了。

异步代码（`async/await`）让程序在**等待 IO 的时候可以做别的事**：

```
同步：[---等LLM---][处理][---等LLM---][处理]
异步：[---等LLM---]       → 空出来处理其他任务
         [---等LLM---]    → 另一个请求也在等
```

OpenAI SDK 提供了 `AsyncOpenAI`，用法和同步版几乎一样，只是方法前加 `await`。

In [ ]:
import asyncio
from openai import AsyncOpenAI

async def simple_chat(user_message: str) -> str:
    client = AsyncOpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama",
    )
    
    response = await client.chat.completions.create(
        model="gpt-oss:120b",
        messages=[{"role": "user", "content": user_message}],
    )
    
    await client.close()  # 记得关闭连接
    return response.choices[0].message.content

# Jupyter 里可以直接 await（已经在事件循环里了）
result = await simple_chat("Python 里 list 和 tuple 的区别是什么？")
print(result)

## 4. 流式输出

现在你等着 LLM 把整段话生成完再给你——可能要等 10-20 秒。

流式输出（streaming）让模型**生成一个词就发一个词**，用户体验好很多。就像 Claude / ChatGPT 那样一个字一个字打出来。

原理：把 `stream=True` 传给 API，返回的不是一个完整 response，而是一个**异步生成器**，不断 yield 出 chunk（数据块）。

每个 chunk 的结构：
```
chunk.choices[0].delta.content  →  这次新增的文字（可能是一个词，可能是几个字）
chunk.choices[0].finish_reason  →  None（还在生成）或 "stop"（结束了）
```

In [ ]:
import asyncio
from openai import AsyncOpenAI

async def stream_chat(user_message: str):
    client = AsyncOpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama",
    )
    
    stream = await client.chat.completions.create(
        model="gpt-oss:120b",
        messages=[{"role": "user", "content": user_message}],
        stream=True,  # 开启流式
    )
    
    full_text = ""
    
    async for chunk in stream:  # 异步迭代每个数据块
        delta = chunk.choices[0].delta.content  # 这次新增的文字
        
        if delta:  # 有时候 delta 是 None（比如第一个 chunk）
            print(delta, end="", flush=True)  # flush=True 确保立刻打印
            full_text += delta
    
    print()  # 换行
    await client.close()
    return full_text

result = await stream_chat("用 50 字解释什么是闭包")
print(f"\n[共 {len(result)} 个字符]")

## 5. 指数退避重试

网络请求会失败。常见原因：
- `RateLimitError`：请求太频繁，服务限速了
- `APIConnectionError`：网络抖动，连接断了
- `InternalServerError`：服务器临时出错（500）

这些错误通常是**临时的**，等一下再试就好了。

**指数退避**：第 1 次失败等 1 秒，第 2 次等 2 秒，第 3 次等 4 秒……

```
失败 → 等 1s → 重试 → 失败 → 等 2s → 重试 → 失败 → 等 4s → 重试 → 放弃
```

为什么是指数而不是固定间隔？如果服务器过载，所有客户端同时重试会让情况更糟。指数退避让重试压力分散。

In [ ]:
import asyncio
from openai import AsyncOpenAI, RateLimitError, APIConnectionError, InternalServerError

async def chat_with_retry(
    messages: list,
    model: str = "gpt-oss:120b",
    max_retries: int = 3,
    stream: bool = False,
):
    client = AsyncOpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama",
    )
    
    # 可重试的错误类型
    retryable_errors = (RateLimitError, APIConnectionError, InternalServerError)
    
    for attempt in range(max_retries + 1):  # 0, 1, 2, 3
        try:
            response = await client.chat.completions.create(
                model=model,
                messages=messages,
                stream=stream,
            )
            await client.close()
            return response
            
        except retryable_errors as e:
            if attempt == max_retries:
                # 已经是最后一次，放弃
                await client.close()
                raise
            
            wait_time = 2 ** attempt  # 1, 2, 4 秒
            print(f"[重试 {attempt + 1}/{max_retries}] {type(e).__name__}，等待 {wait_time}s...")
            await asyncio.sleep(wait_time)
            
        except Exception as e:
            # 不可重试的错误（比如参数错误），直接抛出
            await client.close()
            raise


# 测试正常调用
response = await chat_with_retry(
    messages=[{"role": "user", "content": "1+1等于几？"}]
)
print(response.choices[0].message.content)

In [ ]:
# 模拟重试：用一个会先失败的 mock 来验证退避逻辑
from unittest.mock import AsyncMock, patch
import openai

call_count = 0

async def mock_create(*args, **kwargs):
    global call_count
    call_count += 1
    if call_count < 3:  # 前两次失败
        raise APIConnectionError(request=None)
    # 第三次成功（返回一个假 response）
    from openai.types.chat import ChatCompletion, ChatCompletionMessage
    from openai.types.chat.chat_completion import Choice
    from openai.types import CompletionUsage
    return ChatCompletion(
        id="mock",
        choices=[Choice(
            finish_reason="stop",
            index=0,
            message=ChatCompletionMessage(role="assistant", content="重试成功！")
        )],
        created=0,
        model="gpt-oss:120b",
        object="chat.completion",
        usage=CompletionUsage(completion_tokens=3, prompt_tokens=5, total_tokens=8)
    )

with patch("openai.resources.chat.completions.AsyncCompletions.create", side_effect=mock_create):
    call_count = 0
    response = await chat_with_retry(
        messages=[{"role": "user", "content": "test"}],
        max_retries=3
    )
    print(f"最终结果: {response.choices[0].message.content}")
    print(f"总共调用了 {call_count} 次")

## 6. 封装成类

把上面的零散函数整合成一个类。设计目标：

1. **连接复用**：每次调用不重新创建 `AsyncOpenAI`，用一个共享的 client
2. **懒初始化**：第一次用时才创建 client，不用时不占资源
3. **统一接口**：流式和非流式用同一个方法，传 `stream=True/False` 区分
4. **可替换**：换 Anthropic / OpenRouter 只改初始化参数

In [ ]:
import asyncio
from typing import AsyncGenerator
from openai import AsyncOpenAI, RateLimitError, APIConnectionError, InternalServerError
from dataclasses import dataclass


@dataclass
class TokenUsage:
    """记录一次 LLM 调用的 token 消耗"""
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int


class LLMClient:
    """
    异步 LLM 客户端，兼容任何 OpenAI 格式的接口。
    
    用法：
        client = LLMClient()  # 默认连接 Ollama
        
        # 非流式
        text, usage = await client.chat_completion(messages)
        
        # 流式
        async for chunk in client.chat_completion(messages, stream=True):
            print(chunk, end="")
    """

    def __init__(
        self,
        model: str = "gpt-oss:120b",
        base_url: str = "http://localhost:11434/v1",
        api_key: str = "ollama",
        max_retries: int = 3,
        temperature: float = 0.7,
        max_tokens: int = 4096,
    ):
        self.model = model
        self.base_url = base_url
        self.api_key = api_key
        self.max_retries = max_retries
        self.temperature = temperature
        self.max_tokens = max_tokens
        self._client: AsyncOpenAI | None = None  # 懒初始化

    def _get_client(self) -> AsyncOpenAI:
        if self._client is None:
            self._client = AsyncOpenAI(
                base_url=self.base_url,
                api_key=self.api_key,
            )
        return self._client

    async def close(self):
        if self._client is not None:
            await self._client.close()
            self._client = None

    async def chat_completion(
        self,
        messages: list[dict],
        stream: bool = False,
        tools: list[dict] | None = None,  # 工具列表（后面章节用到）
    ):
        """
        发送对话请求。
        
        - stream=False: 返回 (text: str, usage: TokenUsage)
        - stream=True:  返回 AsyncGenerator，每次 yield 一段文字
        """
        if stream:
            return self._stream(messages, tools)
        else:
            return await self._complete(messages, tools)

    async def _complete(
        self,
        messages: list[dict],
        tools: list[dict] | None,
    ) -> tuple[str, TokenUsage]:
        """非流式调用，带重试"""
        retryable = (RateLimitError, APIConnectionError, InternalServerError)
        kwargs = self._build_kwargs(messages, tools, stream=False)

        for attempt in range(self.max_retries + 1):
            try:
                response = await self._get_client().chat.completions.create(**kwargs)
                text = response.choices[0].message.content or ""
                usage = TokenUsage(
                    prompt_tokens=response.usage.prompt_tokens,
                    completion_tokens=response.usage.completion_tokens,
                    total_tokens=response.usage.total_tokens,
                )
                return text, usage

            except retryable as e:
                if attempt == self.max_retries:
                    raise
                wait = 2 ** attempt
                print(f"[LLMClient] 重试 {attempt + 1}/{self.max_retries}，等待 {wait}s ({type(e).__name__})")
                await asyncio.sleep(wait)

    async def _stream(
        self,
        messages: list[dict],
        tools: list[dict] | None,
    ) -> AsyncGenerator[str, None]:
        """流式调用，yield 每个文字片段"""
        kwargs = self._build_kwargs(messages, tools, stream=True)
        stream = await self._get_client().chat.completions.create(**kwargs)

        async for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                yield delta

    def _build_kwargs(self, messages, tools, stream) -> dict:
        kwargs = {
            "model": self.model,
            "messages": messages,
            "stream": stream,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }
        if tools:
            kwargs["tools"] = tools
        return kwargs


print("LLMClient 定义完成")

In [ ]:
# 测试非流式
client = LLMClient()

messages = [
    {"role": "system", "content": "你是一个简洁的技术助手，回答不超过 3 句话。"},
    {"role": "user",   "content": "什么是 Python 的 GIL？"},
]

text, usage = await client.chat_completion(messages, stream=False)

print("回答:")
print(text)
print()
print(f"Token 消耗: {usage.prompt_tokens} 输入 + {usage.completion_tokens} 输出 = {usage.total_tokens} 合计")

In [ ]:
# 测试流式
messages = [
    {"role": "system", "content": "你是一个简洁的技术助手。"},
    {"role": "user",   "content": "解释一下什么是异步编程，用比喻来说明"},
]

print("流式输出：")
print("-" * 40)

async for chunk in await client.chat_completion(messages, stream=True):
    print(chunk, end="", flush=True)

print()
print("-" * 40)

In [ ]:
# 多轮对话测试：验证历史记录正确传递
conversation = [
    {"role": "system", "content": "你是一个技术助手，回答简洁。"}
]

# 第一轮
conversation.append({"role": "user", "content": "我在学 Python，先告诉我什么是列表"})
text, _ = await client.chat_completion(conversation)
conversation.append({"role": "assistant", "content": text})
print("第一轮回答:")
print(text[:200], "...")
print()

# 第二轮：问的是「它和字典的区别」—— 模型需要记住上下文才能正确理解「它」
conversation.append({"role": "user", "content": "好，那它和字典有什么区别？"})
text, usage = await client.chat_completion(conversation)
print("第二轮回答（模型需要根据上下文知道'它'是列表）:")
print(text[:300])
print()
print(f"此时携带了 {len(conversation)+1} 条消息，共消耗 {usage.total_tokens} tokens")

In [ ]:
# 关闭客户端，释放连接
await client.close()
print("客户端已关闭")

## 7. 换接口只需改两个参数

这是用 OpenAI SDK 对接所有兼容接口的好处：

```python
# 对接 Ollama（本地）
client = LLMClient(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    model="gpt-oss:120b",
)

# 对接 OpenRouter（访问 GPT-4、Claude 等）
client = LLMClient(
    base_url="https://openrouter.ai/api/v1",
    api_key="sk-or-xxxx",
    model="anthropic/claude-3.5-sonnet",
)

# 对接 Anthropic 直接 API
client = LLMClient(
    base_url="https://api.anthropic.com/v1",
    api_key="sk-ant-xxxx",
    model="claude-sonnet-4-6",
)
```

后面所有章节都用 `LLMClient` 这个类，不会再重复写 `AsyncOpenAI`。

## 8. 小结

这章做了什么：

| 概念 | 关键点 |
|------|--------|
| messages 格式 | role: system/user/assistant，LLM 无状态，每次把全部历史带过去 |
| Ollama 对接 | 只需要改 `base_url`，其他和 OpenAI API 完全一样 |
| 异步 | `AsyncOpenAI` + `await`，等 IO 时不阻塞 |
| 流式 | `stream=True`，async for 迭代 chunk，实时打印 |
| 重试 | 指数退避（1s/2s/4s），只重试可恢复的错误 |
| 封装 | `LLMClient` 类，懒初始化，连接复用 |

**下一章：Context Manager**

现在你能发消息、拿回答了，但还需要管理对话历史——消息太多了怎么办？token 超限了怎么办？系统 prompt 怎么构造？这些在第 02 章解决。

---

## 附：把 LLMClient 保存为独立模块

后面的章节会 import 这个类，先把它存成文件。

In [ ]:
import pathlib

src_dir = pathlib.Path("src")
src_dir.mkdir(exist_ok=True)
(src_dir / "__init__.py").touch()

llm_client_code = '''
import asyncio
from typing import AsyncGenerator
from dataclasses import dataclass
from openai import AsyncOpenAI, RateLimitError, APIConnectionError, InternalServerError


@dataclass
class TokenUsage:
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int


class LLMClient:
    def __init__(
        self,
        model: str = "gpt-oss:120b",
        base_url: str = "http://localhost:11434/v1",
        api_key: str = "ollama",
        max_retries: int = 3,
        temperature: float = 0.7,
        max_tokens: int = 4096,
    ):
        self.model = model
        self.base_url = base_url
        self.api_key = api_key
        self.max_retries = max_retries
        self.temperature = temperature
        self.max_tokens = max_tokens
        self._client: AsyncOpenAI | None = None

    def _get_client(self) -> AsyncOpenAI:
        if self._client is None:
            self._client = AsyncOpenAI(base_url=self.base_url, api_key=self.api_key)
        return self._client

    async def close(self):
        if self._client is not None:
            await self._client.close()
            self._client = None

    async def chat_completion(self, messages: list[dict], stream: bool = False, tools: list[dict] | None = None):
        if stream:
            return self._stream(messages, tools)
        return await self._complete(messages, tools)

    async def _complete(self, messages, tools) -> tuple[str, TokenUsage]:
        retryable = (RateLimitError, APIConnectionError, InternalServerError)
        kwargs = self._build_kwargs(messages, tools, stream=False)
        for attempt in range(self.max_retries + 1):
            try:
                response = await self._get_client().chat.completions.create(**kwargs)
                text = response.choices[0].message.content or ""
                usage = TokenUsage(
                    prompt_tokens=response.usage.prompt_tokens,
                    completion_tokens=response.usage.completion_tokens,
                    total_tokens=response.usage.total_tokens,
                )
                return text, usage
            except retryable as e:
                if attempt == self.max_retries:
                    raise
                wait = 2 ** attempt
                print(f"[LLMClient] 重试 {attempt + 1}/{self.max_retries}，等待 {wait}s")
                await asyncio.sleep(wait)

    async def _stream(self, messages, tools) -> AsyncGenerator[str, None]:
        kwargs = self._build_kwargs(messages, tools, stream=True)
        stream = await self._get_client().chat.completions.create(**kwargs)
        async for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                yield delta

    def _build_kwargs(self, messages, tools, stream) -> dict:
        kwargs = dict(
            model=self.model, messages=messages, stream=stream,
            temperature=self.temperature, max_tokens=self.max_tokens,
        )
        if tools:
            kwargs["tools"] = tools
        return kwargs
'''

with open("src/llm_client.py", "w") as f:
    f.write(llm_client_code.strip())

print("已保存到 src/llm_client.py")
print("后续章节用法: from src.llm_client import LLMClient, TokenUsage")